# Setup

## Imports

In [ ]:
# matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# others
import airr
import colorspacious as cs
import numpy as np
import pandas as pd
import seaborn as sns

# add repertoire_zoo
import repertoire_zoo.bird as br
import repertoire_zoo.giraffe as gr

## Setup directories - Read in data

In [ ]:
# Define project/job uuid
project_id = ... # project uuid
job_id = ... # jov uuid

# pull out repertoires and names
groups_of_interest = [ ... ] # insert groups to compare (to compare all replace `...` with `None`)

# identify control group(s) to compare against
control_group_names = [ ... ] # insert control groups (for no control replace `...` with `None`)

# Define data directories
data_dir = ... #/path/to/data/directory/
repcalc_dir = data_dir + 'vdjserver/'+job_id+'/'
analysis_dir = data_dir + 'analysis/'
fig_dir = analysis_dir + 'figures/'

# Read in repertoire and repertoire group info
# The default name (repertoires.airr.json) is being used, but custom airr.json files can
# be used for defining repertoires and repertoire groups. 
repertoires_airr_json_filename = 'repertoires.airr.json'
sample_info_df = br.read_repertoire_info_airr(repcalc_dir + repertoires_airr_json_filename)
repertoire_group_df = br.read_repertoire_group_info_airr(repcalc_dir + repertoires_airr_json_filename)
data = airr.read_airr(repcalc_dir + repertoires_airr_json_filename)

if groups_of_interest == [None]:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] }
else:
    repertoire_groups_temp = { obj['repertoire_group_id'] : obj for obj in data['RepertoireGroup'] if obj['repertoire_group_name'] in groups_of_interest }
repertoires = { obj['repertoire_id'] : obj for obj in data['Repertoire']}

order_map = {name: idx for idx, name in enumerate(groups_of_interest)}
repertoire_groups = dict(sorted(
    repertoire_groups_temp.items(),
    key=lambda item: order_map.get(item[1]['repertoire_group_name'], len(groups_of_interest))
))

# make groups tuple
groups_tuple = [
    (
        repertoire_groups[rep_group_id]['repertoire_group_id'],
        repertoire_groups[rep_group_id]['repertoire_group_name'],
        [reps.get('repertoire_id') for reps in repertoire_groups[rep_group_id]['repertoires']]
    ) for rep_group_id in repertoire_groups
]

for i, rep_group_id in enumerate(repertoire_groups):
    print(rep_group_id, i, repertoire_groups[rep_group_id]['repertoire_group_name'])

condition_counts = {e[1] : e[2] for e in groups_tuple}

## Colormaps

In [ ]:
# set hue order (use condition names)
# By default the hue_order will be the same as the groups of interest. 
# If this is different, redfine `hue_order` as a list of groups from `groups_of_interest`
hue_order = groups_of_interest

cmap = mpl.colormaps['nipy_spectral'] # gist_rainbow
cmin = 0.0
cmax = 1.0

# V
v_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='gene'
)['gene'].unique()
v_genes_cmap = {
    v_gene : cmap(np.linspace(cmin,cmax,len(v_genes))[i])
    for i, v_gene in enumerate(v_genes)
}

v_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='v_call',
    level='subgroup'
)['gene'].unique()
v_subgroup_cmap = {
    v_subgroup : cmap(np.linspace(cmin,cmax,len(v_subgroups))[i])
    for i, v_subgroup in enumerate(v_subgroups)
}

# J
j_genes = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='gene'
)['gene'].unique()
j_genes_cmap = {
    j_gene : cmap(np.linspace(cmin,cmax,len(j_genes))[i])
    for i, j_gene in enumerate(j_genes)
}

j_subgroups = br.load_gene_usage_group_data(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    call_type='j_call',
    level='subgroup'
)['gene'].unique()
j_subgroup_cmap = {
    j_subgroup : cmap(np.linspace(cmin,cmax,len(j_subgroups))[i])
    for i, j_subgroup in enumerate(j_subgroups)
}

## cond_cmap
conds = br.load_and_prepare_data_junction_aa(
    repcalc_dir=repcalc_dir,
    groups=groups_tuple,
    processing_stage='.igblast.makedb.allele.clone.group.',
    call_type='junction_aa_length'
)['condition'].unique()

# Colorblind safe colors and colorblind analysis
# number of colors = number of base groups to compare
base_colors = ['#A577F9', '#699B59', '#D36135', '#086788', '#FFBC42']
cond_cmap = {
    cond : base_colors[i]
    for i, cond in enumerate(conds)
}

names = [cond[1] for cond in groups_tuple]

deuteranomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "deuteranomaly",
        "severity": 100
    },
    end = "sRGB1"
)

protanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "protanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

tritanomaly_simulated_cmap = cs.cspace_convert(
    [mcolors.to_rgb(e) for e in cond_cmap.values()], 
    start = {
        "name": "sRGB1+CVD",
        "cvd_type": "tritanomaly",
        "severity": 100
    },
    end = "sRGB1"
)

fig, axes = plt.subplots(4, 1, figsize=(len(list(cond_cmap.keys())), 4))
colorblind_simulated_cmaps = [deuteranomaly_simulated_cmap, protanomaly_simulated_cmap, tritanomaly_simulated_cmap]
gr.show_colormap(cond_cmap, axes[0], False)
for i, (ax, colorblind_simulated_cmap) in enumerate(zip(axes[1:], colorblind_simulated_cmaps)):
    colorblind_simulated_cmap = [np.clip(e, 0, 1) for e in colorblind_simulated_cmap]
    colorblind_simulated_cmap = {k:v for k,v in zip(cond_cmap.keys(), colorblind_simulated_cmap)}
    if i != 2: gr.show_colormap(colorblind_simulated_cmap, ax, False)
    else: gr.show_colormap(colorblind_simulated_cmap, ax, True, 2.5)

axes[0].set_ylabel('Normal', rotation=0, labelpad=20)
axes[1].set_ylabel('Deuteranomaly', rotation=0, labelpad=39)
axes[2].set_ylabel('Protanomaly', rotation=0, labelpad=32)
axes[3].set_ylabel('Tritanomaly', rotation=0, labelpad=32)
fig.suptitle('Colorblind Analysis', fontsize=24, y=1.02)
fig.show()

# Clonality

## Abundance

In [ ]:
processing_stage = 'igblast.makedb.allele.clone'
clone_tool = 'repcalc' # repcalc, changeo

df = br.load_and_prepare_clonal_abundance_group_data(
    repcalc_dir,
    groups_tuple,
    all_groups=repertoire_groups,
    all_repertoires=repertoires,
    clone_tool=clone_tool,
    processing_stage=processing_stage
)

fig, ax = gr.plot_clonal_abundance(
    df,
    y_col = 'abundance',
    figsize=(16, 12),
    title = f'Clonal Abundance Tool Type: {clone_tool}, Filter: Group',
    legend_title = 'Group Name',
    hue='condition'
)

gr.add_N_to_legend(ax, groups_tuple)

fig.show()

## Cumulative abundance

In [ ]:
processing_stage = 'igblast.makedb.allele.clone'
clone_tool = 'repcalc' # repcalc, changeo

nrows = 1
ncols = 3

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(8*ncols, 4*nrows),
    sharex=True,
    sharey=True
)

for i, group in enumerate(groups_tuple):
    if type(axes) == np.ndarray:
        if i==0:
            axes = axes.flatten()
        ax = axes[i]
    else:
        ax = axes
    
    df = br.load_and_prepare_clonal_abundance_group_data(
        repcalc_dir,
        [group],
        all_groups=repertoire_groups,
        all_repertoires=repertoires,
        clone_tool=clone_tool,
        processing_stage=processing_stage
    )

    gr.plot_clonal_abundance(
        df[df['condition']==group[1]],
        y_col = 'cumulative_abundance',
        title = f'{group[1]}',
        legend_title = 'Sample',
        hue='sample_id',
        ax=ax
    )

for ax in axes:
    ax.set_ylim(-0.05,1.05)
    ax.set_xlim(0.9e0, 3e4)
    ax.get_legend().remove()

fontsize=int(max(24, 1.25 * fig.get_size_inches()[0]))
fig.suptitle(**{'t':'Cumulative Abundance','y':1.0, 'fontsize':fontsize})

plt.tight_layout()
fig.show()

## Expansion

In [ ]:
# processing_stage = 'igblast.makedb.allele.clone'
# clone_tool = 'repcalc' # repcalc, changeo

# df = br.load_and_prepare_clonal_abundance_group_data(
#     repcalc_dir,
#     diag_hi_groups_3t7,
#     all_groups=repertoire_groups,
#     all_repertoires=repertoires,
#     clone_tool=clone_tool,
#     processing_stage=processing_stage
# )

# # Assign a distinct color to each category
# unique_labels = df['condition'].unique()
# palette = sns.color_palette('tab10', len(unique_labels))
# color_map = dict(zip(unique_labels, palette))

# plt.figure(figsize=(12, 8))
# for label in unique_labels:
#     subset = df[(df['condition']==label)]
#     subset = subset.sort_values('abundance', ascending=False)
#     subset['group_rank'] = range(1, subset.shape[0]+1)
#     subset = subset[subset['group_rank']<6]

#     plt.scatter(
#         subset['abundance'], subset['seq_freq'],
#         s=(1/(subset['group_rank']**2))*1000,          # scale size if needed
#         color=cond_cmap[label],
#         alpha=0.7,
#         edgecolor='k',
#         label=label
#     )
# plt.xlabel('Abundance')
# plt.ylabel('Sequence Freq')
# plt.legend(title='Condition (Color) / Rank (Size)', bbox_to_anchor=(1.0,0.5))
# plt.title('Test Figure')
# # plt.tight_layout()

In [ ]:
# percentage = False
# similarity_metrics = ['simpson_coefficient', 'jaccard']
# tissue_types = ['cancer_vs_benign', 'all_tissue', 'all_blood', 'tissue_vs_blood']
# tissue_types = ['cancer_vs_benign']
# sharings = ['cdr3_aa_sharing', 'cdr3_nucleotide_sharing', 'cdr3_vj_aa_sharing']
# sharing = sharings[0]


# df['ss_id'] = str(df['clone_id']) + "_" + df['sample_id'] + "_" +  df['condition']
# # old_subjects = ['13N-1', '16M', '17M', '17N', '19M', '19N-1', '46150-1', '46346-1']
# # old_subjects = ['13N-1', '16M', '17M', '17N', '19M', '19N-1']
# #removing old samples from library 25 and 26
# # selected_samples = sample_info_df[~(sample_info_df.subject_id.isin(old_subjects))]
# # selected_samples = selected_samples.sort_values(by = ['subject_id','diagnosis', 'sample_id', 'tissue'])

# repertoire_ids = df.repertoire_id.values
# sample_ids = df.ss_id.values

# path = f"{repcalc_dir}{processing_stage}.{sharing}.tsv"
# df = pd.read_csv(path, sep='\t', index_col=0)
# df = df.loc[repertoire_ids, repertoire_ids]
# df.index = sample_ids
# df.columns = sample_ids

# common_matrix = df.copy()
# N = common_matrix.shape[0]
# pairs = []

# for i in range(N):
#     for j in range(i+1, N):  # upper triangle, no diagonal
#         A_count = common_matrix.iloc[i, i]
#         B_count = common_matrix.iloc[j, j]
#         SampleA_name = common_matrix.index[i]
#         SampleB_name = common_matrix.index[j]
#         common = common_matrix.iloc[i, j]
#         overlap_percent_j = common /((A_count+ B_count)-common) * 100 if min(A_count, B_count) > 0 else 0
#         overlap_percent_s = common /min(A_count, B_count)* 100 if min(A_count, B_count) > 0 else 0
#         overlap_percent_a_in_b = common / A_count * 100 if min(A_count, B_count) > 0 else 0
#         overlap_percent_b_in_a = common / B_count * 100 if min(A_count, B_count) > 0 else 0
#         pairs.append({
#             'SampleA_count': A_count,
#             'SampleB_count': B_count,
#             'Common_count': common,
#             'Overlap_percent_jaccard': overlap_percent_j,
#             'Overlap_percent_simpson': overlap_percent_s,
#             'overlap_percent_a_in_b': overlap_percent_a_in_b,
#             'overlap_percent_b_in_a': overlap_percent_b_in_a,
#             'SampleA_name': SampleA_name,
#             'SampleB_name': SampleB_name
#         })

# df_pairs = pd.DataFrame(pairs)
# df_pairs = df_pairs[df_pairs.Overlap_percent_jaccard > 10]
# print(df_pairs.shape)
# print(df_pairs)
# # Plot
# plt.figure(figsize=(20,16))
# scatter = plt.scatter(
#     df_pairs['SampleA_count'], 
#     df_pairs['SampleB_count'], 
#     s=df_pairs['Common_count'],  
#     c=df_pairs['Overlap_percent_jaccard'], 
#     cmap='viridis', 
#     alpha=0.7, 
#     edgecolors='k'
# )

# plt.colorbar(scatter, label='Overlap % (relative to smaller sample)')
# plt.xlabel('Rearrangement Count Sample A')
# plt.ylabel('Rearrangement Count Sample B')
# plt.title('Scatter Plot of Sample Sizes vs. Overlap')
# plt.grid(True)

# # Add annotations for each point
# for idx, row in df_pairs.iterrows():
#     label = f"{row['SampleA_name']} - {row['SampleB_name']}"
#     plt.annotate(label, (row['SampleA_count'], row['SampleB_count']),
#                  textcoords="offset points", xytext=(5,-5), ha='right', fontsize=8)

# plt.show()

## Diversity curve

In [ ]:
dfs = []
for i, (group_id, group_name, *_) in enumerate(groups_tuple):
        
    df = br.load_diversity_group_data(
        data_dir=repcalc_dir,
        repertoire_group=repertoire_groups[group_id],
        processing_stage='igblast.makedb.gene.clone'
    )
    dfs.append(df)
df = pd.concat(dfs)

fig, ax = gr.plot_diversity_curve(
    df=df,
    title='Diversity Curve',
    legend=True
)

gr.add_N_to_legend(ax, groups_tuple)

fig.show()

## Normalized Shannon entropy

In [ ]:
df = br.load_diversity_multiple_group_data(
    data_dir=repcalc_dir,
    repertoire_groups=groups_tuple,
    processing_stage='igblast.makedb.gene.clone',
    all_groups=repertoire_groups
)

fig, ax = gr.plot_diversity_boxplot(
    df=df,
    hue='condition',
    legend=False
)

gr.add_N_to_xaxis(ax, groups_tuple)

fig.show()